# Chapter 7 — Exporting the scored candidates from chapter 5

*Companion notebook for* **AI Recommender Systems** *(Manning), Chapter 7. Run this before `ch07_ordering.ipynb`.*

The ordering model is **stacked** on the chapter-5 scorer: the cross-encoder score is one of its features. Stacking needs two things chapter 5's two-way split can't give:

1. training labels for the ranker that are disjoint from its test labels, both later than the history the features see;
2. an upstream score on the ranker's training rows from a scorer that never saw those labels. Otherwise the ranker learns to trust an in-sample score, and over-trusts it at test time.

So each user's positive history is cut three ways, per user and in time order, exactly as chapter 5 cuts it in two:

| slice | share of each user's positives | used for |
|---|---|---|
| early | 0–70% | upstream models for the ranker's **training** rows |
| mid | 70–80% | the ranker's **training labels** |
| late | 80–100% | the ranker's **test labels** = chapter 5's test set |

The chapter-5 models are fit twice: on *early* (for the training rows) and on *early + mid*, which is chapter 5's own training set (for the test rows). The 80% cut is checked below to reproduce `temporal_split_per_user` row for row, so the test rows are graded on chapter 5's ground truth and the scored-order baseline **is** chapter 5's result.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "recsys").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import json

import numpy as np
import pandas as pd
import torch

from recsys.data.loaders import load_movielens
from recsys.data.preprocessing import (
    add_item_idx, build_item_index, filter_min_item_ratings, filter_positive,
    sample_active_users, temporal_split_per_user,
)
from recsys.evaluation.metrics import ndcg_at_k as ch5_ndcg_at_k
from recsys.fourstage_recsys.ordering import (
    make_synthetic_movielens, per_user_temporal_slices,
)
from recsys.fourstage_recsys.ordering.upstream import fit_upstream, score_candidates

USE_SYNTHETIC = False    # True -> small synthetic MovieLens-schema run (minutes, CPU)
SEED = 42                # chapter 5's seed: same user sample, same split
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
K_RETRIEVE = 100         # chapter 5's pipeline retrieves 100 candidates per request

CH5_ART = project_root / "data" / "processed" / "chapter05"
ART = project_root / "data" / "processed" / "chapter07"
ART.mkdir(parents=True, exist_ok=True)

# Smaller two-tower settings for the synthetic catalog (a few hundred items);
# None -> chapter 5's hyperparameters unchanged.
INFONCE_OVERRIDES = (dict(emb_dim=32, epochs=4, warmup_epochs=1,
                          pool_start=10, pool_end=100, batch_size=256)
                     if USE_SYNTHETIC else None)

## Data: chapter 5's preprocessing, unchanged

Same user sample, same item filter, same positive threshold. We also keep `ratings_sample`, all ratings of any value for the sampled users and items, because the chapter-7 features (mean rating, rating spread, genre affinity) need the non-positive ratings too.

In [ ]:
if USE_SYNTHETIC:
    ratings, movies = make_synthetic_movielens(seed=7)
else:
    ratings, movies = load_movielens("ml-25m", data_dir=project_root / "data")

ratings_sample = sample_active_users(ratings, n_users=10_000, min_ratings=20, seed=SEED)
ratings_sample = filter_min_item_ratings(ratings_sample, min_ratings=10)
interactions = filter_positive(ratings_sample, threshold=4.0)

item_ids, item_to_idx, idx_to_title, idx_to_genres = build_item_index(interactions, movies)
interactions = add_item_idx(interactions, item_to_idx)
print(f"{interactions.userId.nunique():,} users, {len(item_ids):,} items, "
      f"{len(interactions):,} positives, {len(ratings_sample):,} ratings")

## The three-way split, and the check that the 80% cut is chapter 5's

In [ ]:
early, mid, late = per_user_temporal_slices(interactions, cuts=(0.7, 0.8))

ch5_train, ch5_test = temporal_split_per_user(interactions, test_frac=0.2)
assert pd.concat([early, mid]).index.sort_values().equals(ch5_train.index.sort_values())
assert late.index.sort_values().equals(ch5_test.index.sort_values())
print("80% cut reproduces chapter 5's split exactly.")
print(f"early {len(early):,} | mid {len(mid):,} | late {len(late):,} positives")
print(f"users with ranker-training labels: {mid.userId.nunique():,}; "
      f"with test labels: {late.userId.nunique():,}")

## Ranker training rows: upstream fit on *early*, candidates for users with *mid* labels

This is the extra cost of stacking: one more chapter-5 training run, on less data. Its scores are what the ranker learns to combine; they come from a model that has never seen the labels they are paired with.

In [ ]:
upstream_fit = fit_upstream(early, item_ids, item_to_idx,
                            infonce_params=INFONCE_OVERRIDES, device=DEVICE, seed=SEED)
candidates_fit = score_candidates(upstream_fit, sorted(mid.userId.unique()),
                                  k_retrieve=K_RETRIEVE, device=DEVICE)
print(f"{len(candidates_fit):,} candidate rows for {candidates_fit.userId.nunique():,} users")

## Ranker test rows: upstream fit on *early + mid* (chapter 5's training set)

If chapter 5's two-tower tables are on disk and were built over the same item index, they are reused, so retrieval here is literally chapter 5's retrieval. `03_full_pipeline.ipynb` does not save its cross-encoder, so that is retrained with the same seed. Expect tiny differences from chapter 5's published numbers unless the cross-encoder is saved there too.

In [ ]:
ch5_embeddings = None
if not USE_SYNTHETIC and (CH5_ART / "embeddings.npz").exists():
    with open(CH5_ART / "mappings.json") as fh:
        same_items = json.load(fh)["item_ids"] == list(item_ids)
    if same_items:
        data = np.load(CH5_ART / "embeddings.npz")
        ch5_embeddings = (data["infonce_query"], data["infonce_cand"])
        print("Reusing chapter-5 two-tower embeddings.")
    else:
        print("Chapter-5 item index differs -- retraining the two-tower.")

upstream_test = fit_upstream(pd.concat([early, mid]), item_ids, item_to_idx,
                             embeddings=ch5_embeddings,
                             infonce_params=INFONCE_OVERRIDES, device=DEVICE, seed=SEED)
candidates_test = score_candidates(upstream_test, sorted(late.userId.unique()),
                                   k_retrieve=K_RETRIEVE, device=DEVICE)
print(f"{len(candidates_test):,} candidate rows for {candidates_test.userId.nunique():,} users")

## Wiring check: does the scored order reproduce chapter 5?

Chapter 5's `evaluate_pipeline` scores the first 500 users with the chapter-5 metric. The same computation on the exported candidates, sorted by cross-encoder score, should land on the cross-encoder row of chapter 5's end-to-end table. If it doesn't, stop here: the ordering chapter's baseline would not be chapter 5's.

In [ ]:
relevance = late.groupby("userId")["movieId"].apply(set).to_dict()
eval_users = [u for u in relevance if u in upstream_test.train_items_ids][:500]
top10 = (candidates_test.sort_values(["userId", "cross_encoder_score"],
                                     ascending=[True, False], kind="stable")
         .groupby("userId")["movieId"].apply(lambda s: list(s)[:10]))
ndcgs = [ch5_ndcg_at_k(top10.get(u, []), relevance[u], 10) for u in eval_users]
print(f"Scored order, chapter-5 metric, first {len(eval_users)} users: "
      f"NDCG@10 = {np.mean(ndcgs):.4f}  (compare: 03_full_pipeline, cross-encoder row)")

In [ ]:
ratings_sample.to_parquet(ART / "ratings_sample.parquet", index=False)
movies.to_parquet(ART / "movies.parquet", index=False)
pd.concat([early.assign(slice="early"), mid.assign(slice="mid"),
           late.assign(slice="late")]).to_parquet(ART / "positives.parquet", index=False)
candidates_fit.to_parquet(ART / "candidates_fit.parquet", index=False)
candidates_test.to_parquet(ART / "candidates_test.parquet", index=False)
print("Saved:", sorted(p.name for p in ART.iterdir()))